# Problem 1

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("SparkWhiteBoard").getOrCreate()

data = [(1,"Alice", 25, "NY"),
        (2,"Bob", None, "LA"),
        (3,"Bob", None, "LA"),
        (4,None, 40, "NY")]

df= spark.createDataFrame(data, ["id", "name", "age", "city"])

#1. Remove duplicates based on 'name', 'age', and 'city'
df = df.dropDuplicates(['name', 'age', 'city'])

#2 Fill missing ages with the average age

avg_age = df.select (F.avg("age")).first()[0]
df = df.fillna({"age": avg_age})

#3 Replace missing naes if "Unknown"

df = df.fillna({"name": "Unknown"})

#4 filter people over 30

df = df.filter(F.col("age")>30)

df.show()

spark.stop()

+---+-------+---+----+
| id|   name|age|city|
+---+-------+---+----+
|  2|    Bob| 32|  LA|
|  4|Unknown| 40|  NY|
+---+-------+---+----+



# Problem 2

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("SparkWhiteBoard").getOrCreate()

data = [("A", "Phone", 500),
        ("A", "Laptop", 1000),
        ("B", "Phone", 700)]

df = spark.createDataFrame(data, ["customer", "product", "amount"])


#1 Total spend per customer

total_spend = df.groupBy("customer").agg(F.sum("amount").alias("total_spend"))
total_spend.show()

#2 Average spend per product

avg_spend = df.groupBy("product").agg(F.avg("amount")).alias("average_spend")
avg_spend.show()

#3 Highest spending customewr

Highest_spending = total_spend.orderBy(F.desc("total_spend")).first()[0]

print(Highest_spending)

spark.stop()


+--------+-----------+
|customer|total_spend|
+--------+-----------+
|       A|       1500|
|       B|        700|
+--------+-----------+

+-------+-----------+
|product|avg(amount)|
+-------+-----------+
|  Phone|      600.0|
| Laptop|     1000.0|
+-------+-----------+

A


# Problem 3

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("SparkWhiteBoard").getOrCreate()

data = [("IT",      "John",  100),
        ("IT",      "Mary",  200),
        ("IT",      "Alex",  150),
        ("HR",      "Sara",  300),
        ("HR",      "Tom",   250),
        ("Finance", "Lisa",  400),
        ("Finance", "Dan",   400),   # tie!
        ("Finance", "Ken",   350)]

df = spark.createDataFrame(data, ["department", "employee", "salary"])

from pyspark.sql.window import Window

w = Window.partitionBy("department").orderBy(F.desc("salary"))

df = df.withColumn("rank", F.row_number().over(w))
df.filter(F.col("rank") == 1).drop("rank").show()

spark.stop()


+----------+--------+------+
|department|employee|salary|
+----------+--------+------+
|   Finance|    Lisa|   400|
|        HR|    Sara|   300|
|        IT|    Mary|   200|
+----------+--------+------+



# Problem 4

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("SparkWhiteBoard").getOrCreate()

customers = spark.createDataFrame(
    [(1, "Alice"), (2, "Bob")],
    ["id", "name"]
)

orders = spark.createDataFrame(
    [(1, 100), (1, 50), (3, 25)],
    ["customer_id", "amount"]
)

#1 Inner join

inner_join = customers.join(orders, orders.customer_id == customers.id, how = "inner")

inner_join.show()

#2 Left join

left_join = customers.join(orders, orders.customer_id == customers.id, how = "left")

left_join.show()

#3 Find customers with no orders

customers_with_no_orders = customers.join(orders, orders.customer_id == customers.id, how = "left")

customers_with_no_orders = customers_with_no_orders.filter(F.col("amount").isNull()).select("id", "name")

customers_with_no_orders.show()

#4 Total spend per customer

total_spend = orders.groupBy("customer_id").agg(F.sum("amount").alias("total_spend"))
total_spend = total_spend.join(customers, total_spend.customer_id == customers.id).select("name", "total_spend")
total_spend.show()

spark.stop()



+---+-----+-----------+------+
| id| name|customer_id|amount|
+---+-----+-----------+------+
|  1|Alice|          1|   100|
|  1|Alice|          1|    50|
+---+-----+-----------+------+

+---+-----+-----------+------+
| id| name|customer_id|amount|
+---+-----+-----------+------+
|  1|Alice|          1|    50|
|  1|Alice|          1|   100|
|  2|  Bob|       NULL|  NULL|
+---+-----+-----------+------+

+---+----+
| id|name|
+---+----+
|  2| Bob|
+---+----+

+-----+-----------+
| name|total_spend|
+-----+-----------+
|Alice|        150|
+-----+-----------+



# problem 5

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("SparkWhiteBoard").getOrCreate()

df = spark.createDataFrame(
    [('a@test.com',), ('b@test.com',), ('a@test.com',), ('c@test.com',)],
    ["email"]
)

#1 find the duplicates

dff = df.groupBy("email").count() # df now contains 'email' and 'count' columns
dff = dff.filter(dff['count'] > 1) # Now df['count'] correctly references the 'count' column
dff.show()

#2 count occurrences of each email (including non-duplicates)

dfd = df.groupBy("email").count()
dfd.show()

#3 keep only unique rows

dfd = dfd.filter(dfd['count'] == 1)
dfd.show()

#alternaitve

df.dropDuplicates(['email']).show()


spark.stop()

+----------+-----+
|     email|count|
+----------+-----+
|a@test.com|    2|
+----------+-----+

+----------+-----+
|     email|count|
+----------+-----+
|b@test.com|    1|
|a@test.com|    2|
|c@test.com|    1|
+----------+-----+

+----------+-----+
|     email|count|
+----------+-----+
|b@test.com|    1|
|c@test.com|    1|
+----------+-----+

+----------+
|     email|
+----------+
|b@test.com|
|a@test.com|
|c@test.com|
+----------+



# Problem 6

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("SparkWhiteBoard").getOrCreate()

df = spark.createDataFrame(
    [("A", "Jan1", 10),
     ("A", "Jan2", 20),
     ("A", "Jan3", 15)],
    ["user", "date", "amount"]
)

w = Window.partitionBy("user").orderBy("date")

#1 running total

df = df.withColumn("running_total" , F.sum("amount").over(w))

df.show()

#2 Previous transcation amount

df = df.withColumn("previous_amount", F.lag("amount").over(w))

df.show()

#3 Difference from previous transcation

df = df.withColumn("amount_difference", F.col("amount")-F.col("previous_amount"))

df.show()

#4 Rolling 7-day average

w_7day = Window.partitionBy("user").orderBy("date").rowsBetween(-6, 0)
df.withColumn("rolling_7_avg", F.avg("amount").over(w_7day)).show()

spark.stop()

+----+----+------+-------------+
|user|date|amount|running_total|
+----+----+------+-------------+
|   A|Jan1|    10|           10|
|   A|Jan2|    20|           30|
|   A|Jan3|    15|           45|
+----+----+------+-------------+

+----+----+------+-------------+---------------+
|user|date|amount|running_total|previous_amount|
+----+----+------+-------------+---------------+
|   A|Jan1|    10|           10|           NULL|
|   A|Jan2|    20|           30|             10|
|   A|Jan3|    15|           45|             20|
+----+----+------+-------------+---------------+

+----+----+------+-------------+---------------+-----------------+
|user|date|amount|running_total|previous_amount|amount_difference|
+----+----+------+-------------+---------------+-----------------+
|   A|Jan1|    10|           10|           NULL|             NULL|
|   A|Jan2|    20|           30|             10|               10|
|   A|Jan3|    15|           45|             20|               -5|
+----+----+------+----

# Problem 7

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("SparkWhiteBoard").getOrCreate()

from datetime import datetime

df = spark.createDataFrame(
    [("A", datetime(2024, 1, 1, 10, 0)),
     ("A", datetime(2024, 1, 1, 10, 20)),
     ("A", datetime(2024, 1, 1, 11, 30)),  # >30 min gap, new session
     ("A", datetime(2024, 1, 1, 11, 45)),
     ("B", datetime(2024, 1, 1, 9, 0)),
     ("B", datetime(2024, 1, 1, 9, 10))],
    ["user", "timestamp"]
)

w = Window.partitionBy("user").orderBy("timestamp")

df = df.withColumn("prev_timestamp", F.lag("timestamp").over(w))

df = df.withColumn("new_session",
    F.when(
        (F.col("timestamp") - F.col("prev_timestamp")) > F.expr("interval 30 minutes"), 1
    ).otherwise(0)
)

# Problem 8

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("SparkWhiteBoard").getOrCreate()

df = spark.createDataFrame(
    [("Jan", "A", 10),
     ("Jan", "B", 15),
     ("Feb", "A", 12)],
    ["month", "product", "sales"]
)
a_df = df.filter(F.col("product") == "A")
b_df = df.filter(F.col("product") == "B")

a_df= a_df.drop("product")
b_df= b_df.drop("product")

a_df = a_df.withColumnRenamed("sales", "A")
b_df = b_df.withColumnRenamed("sales", "B")

a_df.show()
b_df.show()

a_df = a_df.join(b_df, on = "month", how = "left")

a_df.show()

+-----+---+
|month|  A|
+-----+---+
|  Jan| 10|
|  Feb| 12|
+-----+---+

+-----+---+
|month|  B|
+-----+---+
|  Jan| 15|
+-----+---+

+-----+---+----+
|month|  A|   B|
+-----+---+----+
|  Jan| 10|  15|
|  Feb| 12|NULL|
+-----+---+----+

